## Input
- `tif`, each conatining only 1 channel
- a cellpose classifyer (default or costum trained)

## Outpout
for each provided `tif`:
- segmentation mask as `png`
- segmentation mask as `npy`
- segmentation outline as `txt`
- vis folder with detection visualization for all `tif`s as `png` (good for checking segmentation results)

# 0) Imports and functions

The functions required for this to work are collected in the `pipelines/fish_utils` folder. Download the folder from `/../` in this repository and `sys.path.append(/path/to/fish_utils/)`. You can skip this if you are providing `tif`s.

In [ ]:
import os
from glob import glob

import matplotlib.pyplot as plt
import torch
import cellpose
from cellpose import models,io
from cellpose import plot

# check if we have CP4 -> different model init. needed?
is_cellpose4 = int(cellpose.version[0]) >= 4

In [ ]:
# to do the segmentaion fast work on the gpu

if torch.cuda.is_available():
    device = torch.device('cuda:0')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

device

# 1) Parameters

In [ ]:
# input paths: base in_path plus subdirectory for tiff files (with pattern)
in_path = '/Volumes/nn/Samriddhi/14-07-26/3CS_Human_NO_DOX/'
image_subdirectory = 'tif' # for resaved: "tif"
image_file_pattern = '*_ch2*.tif' # e.g. "*_ch0.tif" to only include images of one channel

# where to save results
out_subdirectory = 'segmentation_nuclei'
visualization_subdirectory = 'vis'

# Cellpose model, either one of the named official models or path to your own

# model_checkpoint = 'nuclei'
# model_checkpoint = "/home/stumberger/.cellpose/models/es_20231026"
# model_checkpoint = '/Users/david/Downloads/cellpose_es_plus_weihua_cells1'

# NOTE: when using cpdino* models, may need pip install git+https://github.com/facebookresearch/dinov3
model_checkpoint = 'cpdino-vitb'

# segmentation parameters
chan = [[0,0]]
diams = 75
min_size = 4000

# for 3D data: sampling in xy / sampling in z (eg. 0.13 / 0.3 = 2.3)
anisotropy = 1.0

# smoothing for 3D flows
flow3D_smooth = 1.0

# save .npy files that can be loaded in Cellpose GUI
# NOTE: uses quite a lot of space, consider skipping
save_for_cp_gui = False


In [ ]:
# model for segmentation
# use pretrained_model parameter for CP4, model_type for CP<4
if is_cellpose4:
    model = models.CellposeModel(pretrained_model=model_checkpoint, device=device)
else:
    model = models.CellposeModel(model_type=model_checkpoint, device=device)

In [ ]:
# in and out paths based on upper directory
files = glob(os.path.join(in_path, image_subdirectory, image_file_pattern))
out_path = f"{in_path}/{out_subdirectory}"

# files

# 2) Segmentation

In [ ]:
#create out directories
os.makedirs(f"{out_path}/{visualization_subdirectory}", exist_ok=True)

# apply to all files
for filename in files:

    img = io.imread(filename)

    # do 3D if data is 3D else do 2D
    do_3D = img.ndim == 3

    name = os.path.basename(filename).rsplit(".", 1)[0]
    out = f"{out_path}/{name}.tif"
    
    masks, flows, styles = model.eval(img, 
                                      do_3D=do_3D,
                                      diameter = diams,
                                      min_size = min_size,
                                      anisotropy = anisotropy,
                                      flow3D_smooth = flow3D_smooth)

    # save results so you can load in gui
    # io.masks_flows_to_seg(img, masks, flows, diams, out)
    if save_for_cp_gui:
        io.masks_flows_to_seg(img, masks, flows, out, diams) 

    # save results as png
    io.save_masks(img, masks, flows, out, tif=True)
    
    # max projection of segmentation for quick visualization
    fig = plt.figure(figsize=(12,5))

    if do_3D:
        plot.show_segmentation(fig, img.max(axis=0), masks.max(axis=0), flows[0].max(axis=0), channels=chan)
    else:
        plot.show_segmentation(fig, img, masks, flows[0], channels=chan)

    plt.tight_layout()
    fig.savefig(f"{out_path}/{visualization_subdirectory}/{os.path.basename(out)}.png",dpi=300)
    plt.close(fig)